# LeetCode 135 — Candy (Hard) bằng Python

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def candy(self, ratings: List[int]) -> int:

```

Bài này nghe như một câu đố mẫu giáo: chia kẹo cho các bạn nhỏ đứng thành hàng. Nhưng nó là **Hard** vì mỗi bạn phải "so bì" với **cả hai bạn đứng cạnh**: sửa cho bạn bên trái hài lòng có thể làm bạn bên phải mất lòng. Chìa khóa của bài là **tách hai phía ra xử lý riêng** rồi ghép lại.

---

## Sau notebook này, bạn sẽ nắm được

1. Đọc đề thật kỹ: hai luật chia kẹo, và vì sao hai bạn **bằng điểm** thì **không** cần bằng kẹo.
2. Vẽ dãy điểm thành **hình đồi dốc** (lên dốc, xuống dốc, đất bằng) để nhìn ra cấu trúc bài toán.
3. **Cách 0 — Vét cạn** (thử mọi cách chia) để có "trọng tài" kiểm tra đáp án trên input nhỏ.
4. **Cách 1 — Lặp sửa lỗi cho tới khi ổn định** (relaxation) `O(n²)`: cách người mới hay tự nghĩ ra.
5. **Cách 2 — Hai mảng `left`/`right`, hai lượt quét, lấy `max`** `O(n)`: cách chính, kèm giải thích vì sao `max` vừa **đúng luật** vừa **ít kẹo nhất**.
6. **Cách 3 — Một mảng, hai lượt quét**: gọn bộ nhớ hơn cách 2.
7. **Cách 4 — Một lượt quét, bộ nhớ `O(1)`**, chỉ nhớ độ dài dốc lên / dốc xuống / đỉnh (`up` / `down` / `peak`).
8. **Cách 5 — Sắp xếp theo điểm rồi phát kẹo** `O(n log n)`: một góc nhìn khác.
9. Lỗi hay gặp, mẹo nhận dạng pattern, bài liên quan và 6 bài tập có đáp án.

## 1. Đề bài (theo LeetCode)

Có `n` bạn nhỏ đứng thành **một hàng**. Mỗi bạn được chấm một điểm (rating), cho trong mảng số nguyên `ratings`.

Bạn phát kẹo cho các bạn nhỏ theo hai yêu cầu:

- Mỗi bạn phải có **ít nhất 1** viên kẹo.
- Bạn nào có điểm **cao hơn** bạn đứng **cạnh** mình (neighbor — hàng xóm, tức bạn ngay bên trái hoặc ngay bên phải) thì phải được **nhiều kẹo hơn** bạn đó.

Trả về **số kẹo tối thiểu** cần chuẩn bị để phát đúng luật.

**Example 1**

```text
Input:  ratings = [1,0,2]
Output: 5
Explanation: You can allocate to the first, second and third child with 2, 1, 2 candies respectively.
```

**Example 2**

```text
Input:  ratings = [1,2,2]
Output: 4
Explanation: You can allocate to the first, second and third child with 1, 2, 1 candies respectively.
The third child gets 1 candy because it satisfies the above two conditions.
```

**Constraints**

- `n == ratings.length`
- `1 <= n <= 2 * 10^4`
- `0 <= ratings[i] <= 2 * 10^4`

Ba điều cần để ý ngay:

- Hàm trả về **một số nguyên** (tổng số kẹo), không phải danh sách kẹo của từng bạn.
- Luật chỉ so sánh với **hàng xóm sát bên**, không so với cả hàng.
- `n` tới `20 000`: lời giải `O(n²)` cỡ `4 × 10⁸` bước sẽ quá chậm trong Python, ta cần `O(n)` hoặc `O(n log n)`.

In [1]:
from typing import List   # LeetCode tự import sẵn; chạy ngoài LeetCode thì ta tự import

ratings1 = [1, 0, 2]
ratings2 = [1, 2, 2]

print("Example 1:", ratings1, "-> mong đợi 5, ví dụ chia [2, 1, 2]")
print("Example 2:", ratings2, "-> mong đợi 4, ví dụ chia [1, 2, 1]")
print("Tổng kiểm tra:", sum([2, 1, 2]), sum([1, 2, 1]))

Example 1: [1, 0, 2] -> mong đợi 5, ví dụ chia [2, 1, 2]
Example 2: [1, 2, 2] -> mong đợi 4, ví dụ chia [1, 2, 1]
Tổng kiểm tra: 5 4


## 2. Hiểu đề bằng tay trước khi code

Ta viết lại luật thứ hai thành hai câu "nếu… thì…" cho từng vị trí `i` (`candies[i]` là số kẹo của bạn thứ `i`):

- **Luật trái:** nếu `ratings[i] > ratings[i-1]` thì `candies[i] > candies[i-1]`.
- **Luật phải:** nếu `ratings[i] > ratings[i+1]` thì `candies[i] > candies[i+1]`.

Không có luật nào nói về trường hợp **bằng điểm**. Vậy:

- Example 2 `[1, 2, 2]`: bạn 1 và bạn 2 cùng điểm 2. Bạn 1 phải hơn bạn 0 (vì `2 > 1`) nên được 2 kẹo. Bạn 2 **không** cần hơn bạn 1 và cũng không cần bằng bạn 1 → cho 1 kẹo là đủ. Tổng `1 + 2 + 1 = 4`.
- Nếu ta lỡ "công bằng" cho bạn 2 cũng 2 kẹo thì vẫn đúng luật, nhưng tốn `5` kẹo — **không tối thiểu**.

Ví dụ thêm: `ratings = [1, 0, 2]`.

- Bạn 1 (điểm 0) thấp nhất → 1 kẹo.
- Bạn 0 (điểm 1) cao hơn bạn 1 → phải hơn 1 → 2 kẹo.
- Bạn 2 (điểm 2) cao hơn bạn 1 → phải hơn 1 → 2 kẹo. Bạn 2 **không đứng cạnh** bạn 0 nên không cần hơn bạn 0.

Cell dưới viết một hàm **kiểm tra luật** `is_valid(ratings, candies)`. Ta sẽ dùng nó suốt notebook để tự tin rằng mọi cách chia đều hợp lệ.

In [2]:
def is_valid(ratings, candies):
    """True nếu cách chia candies thỏa cả hai luật của đề."""
    n = len(ratings)
    if len(candies) != n:
        return False
    for i in range(n):
        if candies[i] < 1:                                       # luật 1: ít nhất 1 kẹo
            return False
        if i > 0 and ratings[i] > ratings[i - 1] and candies[i] <= candies[i - 1]:
            return False                                         # vi phạm luật trái
        if i < n - 1 and ratings[i] > ratings[i + 1] and candies[i] <= candies[i + 1]:
            return False                                         # vi phạm luật phải
    return True


print(is_valid([1, 0, 2], [2, 1, 2]))   # True
print(is_valid([1, 0, 2], [1, 1, 2]))   # False: bạn 0 điểm cao hơn bạn 1 mà kẹo không nhiều hơn
print(is_valid([1, 2, 2], [1, 2, 1]))   # True: bằng điểm thì không ràng buộc
print(is_valid([1, 2, 2], [1, 2, 2]))   # True nhưng tốn hơn (5 kẹo)

True
False
True
True


## 3. Vẽ dãy điểm thành "đồi dốc"

Mẹo quan trọng nhất của bài này là **đừng nhìn giá trị tuyệt đối** của điểm. Điểm 87 hay điểm 3 không quan trọng; chỉ quan trọng bạn **cao hơn, thấp hơn hay bằng** bạn bên cạnh. Ta vẽ dãy điểm như một con đường:

- `ratings[i] > ratings[i-1]` → đường **đi lên** một bậc (lên dốc),
- `ratings[i] < ratings[i-1]` → đường **đi xuống** một bậc (xuống dốc),
- bằng nhau → đường **đi ngang** (đất bằng).

Ví dụ `[1, 3, 4, 5, 2]`: lên, lên, lên, xuống:

```text
               5
           4       2
       3
   1
   0   1   2   3   4   <- index
     <   <   <   >   <- so với bạn bên trái
```

Ở đây mỗi bạn được vẽ đúng "độ cao bậc" của mình, số in ra là điểm thật. Bạn `5` đứng trên **đỉnh dốc** (peak), cần nhiều kẹo hơn cả hai bên. Bạn `1` ở **đáy** (valley), chỉ cần 1 kẹo.

Cell dưới có hai hàm vẽ:

- `draw_hill(ratings)` vẽ đồi dốc như trên,
- `draw_candies(candies)` vẽ số kẹo thành cột `#` để mắt so sánh nhanh.

Bạn **không cần** hiểu hết chi tiết hai hàm vẽ; chỉ cần biết cách gọi.

In [3]:
def draw_hill(ratings):
    """Vẽ ratings thành đồi dốc: lên 1 bậc nếu tăng, xuống 1 bậc nếu giảm, đi ngang nếu bằng."""
    n = len(ratings)
    level = [0]
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            level.append(level[-1] + 1)
        elif ratings[i] < ratings[i - 1]:
            level.append(level[-1] - 1)
        else:
            level.append(level[-1])
    for lv in range(max(level), min(level) - 1, -1):          # vẽ từ trên xuống
        row = "".join(f"{ratings[i]:>4}" if level[i] == lv else "    " for i in range(n))
        print(row.rstrip())
    print("".join(f"{i:>4}" for i in range(n)) + "   <- index")
    rel = [" "] * (4 * n)
    for i in range(1, n):
        rel[4 * i + 1] = "<" if ratings[i] > ratings[i - 1] else (">" if ratings[i] < ratings[i - 1] else "=")
    print("".join(rel).rstrip() + "   <- so với bạn bên trái")


def draw_candies(candies):
    """Vẽ số kẹo từng bạn thành cột #."""
    for h in range(max(candies), 0, -1):
        print("".join("   #" if c >= h else "    " for c in candies).rstrip())
    print("".join(f"{c:>4}" for c in candies) + "   <- số kẹo")


draw_hill([1, 3, 4, 5, 2])
print()
draw_candies([1, 2, 3, 4, 1])

               5
           4       2
       3
   1
   0   1   2   3   4   <- index
     <   <   <   >   <- so với bạn bên trái

               #
           #   #
       #   #   #
   #   #   #   #   #
   1   2   3   4   1   <- số kẹo


Thêm hai "địa hình" mà ta sẽ trace kỹ ở các phần sau:

- `[1, 2, 87, 87, 87, 2, 1]`: lên dốc, rồi **đất bằng** ba bạn điểm 87, rồi xuống dốc. Nhìn hình bạn sẽ thấy dù 87 rất lớn, đồi vẫn chỉ cao 2 bậc.
- `[1, 6, 10, 8, 7, 3, 2]`: dốc lên **ngắn** (2 bậc) nhưng dốc xuống **dài** (4 bậc). Đây là ca khiến đỉnh phải được "nâng" thêm kẹo.

In [4]:
draw_hill([1, 2, 87, 87, 87, 2, 1])
print()
draw_hill([1, 6, 10, 8, 7, 3, 2])

          87  87  87
       2               2
   1                       1
   0   1   2   3   4   5   6   <- index
     <   <   =   =   >   >   <- so với bạn bên trái

          10
       6       8
   1               7
                       3
                           2
   0   1   2   3   4   5   6   <- index
     <   <   >   >   >   >   <- so với bạn bên trái


Nhìn hai hình trên, hãy tự đoán trước số kẹo (ta sẽ kiểm chứng sau):

- Với `[1, 2, 87, 87, 87, 2, 1]`: dốc lên cho `1, 2, 3`; bạn 87 ở giữa đứng trên đất bằng, không ai bắt bạn ấy phải hơn → `1`; bạn 87 cuối đứng đầu dốc xuống dài 2 bậc → `3, 2, 1`. Tổng `1+2+3+1+3+2+1 = 13`.
- Với `[1, 6, 10, 8, 7, 3, 2]`: dốc xuống phía sau đỉnh dài 4 bậc, nên các bạn sau đỉnh cần `4, 3, 2, 1`, và đỉnh phải hơn `4` → `5`. Dốc lên chỉ cần `1, 2`. Tổng `1+2+5+4+3+2+1 = 18`.

Quan sát then chốt (ta sẽ chứng minh ở mục 8): **số kẹo của một bạn = độ dài con dốc dài hơn trong hai con dốc dẫn xuống từ bạn ấy (về bên trái và về bên phải), cộng 1**.

## 4. Kiến thức Python nền tảng dùng trong bài

### 4.1. Tạo list toàn số 1 và đọc hàng xóm an toàn

`[1] * n` tạo list gồm `n` số 1 — đúng "mỗi bạn ít nhất 1 kẹo".

Khi đọc hàng xóm `i - 1` và `i + 1`, phải cẩn thận ở hai đầu hàng:

- Bạn `0` **không có** hàng xóm bên trái. Trong Python, `ratings[-1]` **không báo lỗi** mà lặng lẽ trả về phần tử **cuối** mảng — rất nguy hiểm vì code sai mà không biết.
- Bạn `n - 1` không có hàng xóm bên phải; `ratings[n]` thì báo `IndexError`.

Cách an toàn: luôn kèm điều kiện `i > 0` hoặc `i < n - 1`, hoặc cho vòng lặp chạy đúng khoảng (`range(1, n)` khi cần `i - 1`).

In [5]:
n = 4
candies = [1] * n
print("candies ban đầu:", candies)

ratings = [10, 20, 30, 40]
print("ratings[-1] =", ratings[-1], "<- KHÔNG phải hàng xóm trái của bạn 0, mà là phần tử cuối!")

for i in range(1, n):            # i bắt đầu từ 1 nên i - 1 luôn hợp lệ
    print(f"i={i}: so ratings[{i}]={ratings[i]} với ratings[{i-1}]={ratings[i-1]}")

candies ban đầu: [1, 1, 1, 1]
ratings[-1] = 40 <- KHÔNG phải hàng xóm trái của bạn 0, mà là phần tử cuối!
i=1: so ratings[1]=20 với ratings[0]=10
i=2: so ratings[2]=30 với ratings[1]=20
i=3: so ratings[3]=40 với ratings[2]=30


### 4.2. Vòng lặp chạy **ngược** từ phải sang trái

Ở cách 2 và 3 ta cần duyệt từ cuối hàng về đầu hàng. Cú pháp `range(start, stop, step)` với `step = -1`:

- `range(n - 2, -1, -1)` cho `n-2, n-3, ..., 1, 0`.
- `stop = -1` vì `range` **không lấy** giá trị `stop`; muốn dừng sau `0` thì `stop` phải là `-1`.
- Bắt đầu từ `n - 2` (không phải `n - 1`) vì ở mỗi bước ta đọc `i + 1`; bạn cuối cùng không có ai bên phải.

In [6]:
n = 5
print(list(range(n - 2, -1, -1)))   # [3, 2, 1, 0]
print(list(range(n - 2, 0, -1)))    # [3, 2, 1]  <- thiếu 0, lỗi hay gặp!
print(list(reversed(range(n - 1)))) # cách viết khác, cũng ra [3, 2, 1, 0]

[3, 2, 1, 0]
[3, 2, 1]
[3, 2, 1, 0]


### 4.3. `zip`, `max` và `sum` để ghép hai mảng

- `zip(a, b)` đi song song hai list, mỗi bước cho một cặp `(a[i], b[i])`.
- `max(x, y)` lấy số lớn hơn.
- `sum(...)` cộng tất cả. Có thể viết gọn `sum(max(x, y) for x, y in zip(a, b))` — đây là **generator expression** (biểu thức sinh: giống list comprehension nhưng không tạo list trung gian).

In [7]:
a = [1, 2, 3, 4, 1]
b = [1, 1, 1, 2, 1]
print(list(zip(a, b)))
print([max(x, y) for x, y in zip(a, b)])
print(sum(max(x, y) for x, y in zip(a, b)))

[(1, 1), (2, 1), (3, 1), (4, 2), (1, 1)]
[1, 2, 3, 4, 1]
11


### 4.4. `itertools.product` để thử **mọi** tổ hợp

`itertools.product(range(1, 4), repeat=2)` sinh mọi cặp `(x, y)` với `x, y ∈ {1, 2, 3}`: tổng cộng `3² = 9` cặp. Với `repeat = n` ta có `k^n` bộ — tăng **theo hàm mũ**, chỉ dùng được cho `n` rất nhỏ. Ta dùng nó để viết "trọng tài" vét cạn.

In [8]:
import itertools

pairs = list(itertools.product(range(1, 4), repeat=2))
print(len(pairs), pairs)
print("Số bộ khi n = 6, mỗi bạn 1..6 kẹo:", 6 ** 6)

9 [(1, 1), (1, 2), (1, 3), (2, 1), (2, 2), (2, 3), (3, 1), (3, 2), (3, 3)]
Số bộ khi n = 6, mỗi bạn 1..6 kẹo: 46656


## 5. Cách 0 — Vét cạn: thử mọi cách chia (chỉ để kiểm tra)

**Ý tưởng:** thử tất cả cách gán số kẹo cho từng bạn, giữ lại cách **hợp lệ** có tổng **nhỏ nhất**.

Mỗi bạn cần bao nhiêu kẹo là đủ để thử? Không ai cần quá `n` kẹo: con dốc dài nhất cũng chỉ có `n` bạn, và (như ta sẽ thấy) đáp án tối ưu cho mỗi bạn không vượt quá độ dài con dốc + 1 ≤ `n`. Vì vậy thử mỗi bạn từ `1` tới `n` là đủ.

Cách này **cực chậm** (`n^n` bộ), nhưng nó **không cần thông minh gì** nên gần như không thể sai. Ta gọi nó là "trọng tài" để chấm các cách giải sau trên input nhỏ.

In [9]:
def candy_bruteforce(ratings: List[int]) -> int:
    n = len(ratings)
    best = None
    for candies in itertools.product(range(1, n + 1), repeat=n):
        if is_valid(ratings, candies):
            total = sum(candies)
            if best is None or total < best:
                best = total
    return best


print(candy_bruteforce([1, 0, 2]))        # 5
print(candy_bruteforce([1, 2, 2]))        # 4
print(candy_bruteforce([1, 3, 4, 5, 2]))  # 11

5
4
11


Độ phức tạp: `O(n^n · n)` thời gian (mỗi bộ kiểm tra mất `O(n)`). Với `n = 7` đã là `7^7 ≈ 823 543` bộ; với `n = 20 000` thì vô vọng. Ta cần suy nghĩ khôn hơn.

## 6. Thử nghĩ "tham lam" một lượt từ trái sang phải — và vì sao sai

Ý nghĩ đầu tiên của nhiều người: đi từ trái sang phải, ai cao điểm hơn bạn bên trái thì cho thêm 1 kẹo so với bạn bên trái, còn lại cho 1 kẹo.

In [10]:
def candy_left_only(ratings: List[int]) -> List[int]:
    n = len(ratings)
    candies = [1] * n
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            candies[i] = candies[i - 1] + 1
    return candies


c = candy_left_only([1, 0, 2])
print("chia:", c, "tổng:", sum(c), "hợp lệ?", is_valid([1, 0, 2], c))

chia: [1, 1, 2] tổng: 4 hợp lệ? False


Kết quả `[1, 1, 2]` **sai luật**: bạn 0 (điểm 1) cao hơn bạn 1 (điểm 0) nhưng không được nhiều kẹo hơn. Lượt quét trái → phải chỉ "nhìn thấy" hàng xóm **bên trái**; luật phải bị bỏ quên.

Vậy làm sao xử lý cả hai luật? Cách tự nhiên nhất cho người mới: **cứ thấy chỗ nào sai luật thì sửa, sửa đi sửa lại cho tới khi không còn chỗ nào sai**.

## 7. Cách 1 — Lặp sửa lỗi cho tới khi ổn định (relaxation)

**Relaxation** (dịch thoáng: "nới dần", "sửa dần") là kỹ thuật: bắt đầu từ một lời giải thô, mỗi vòng sửa những chỗ vi phạm ràng buộc, lặp tới khi một vòng trọn vẹn **không còn gì để sửa** (gọi là **ổn định**, hay điểm bất động — fixed point).

Thuật toán:

1. `candies = [1] * n` (ai cũng 1 kẹo).
2. Lặp:
   - đặt `changed = False`,
   - duyệt mọi `i`: nếu vi phạm luật trái (`ratings[i] > ratings[i-1]` mà `candies[i] <= candies[i-1]`) thì nâng `candies[i] = candies[i-1] + 1`; tương tự với luật phải. Mỗi lần nâng đặt `changed = True`,
   - nếu cả vòng không nâng ai (`changed` vẫn `False`) → dừng.
3. Trả về `sum(candies)`.

Lưu ý: ta chỉ **nâng** vừa đủ (`+1` so với hàng xóm), không bao giờ hạ kẹo. Chi tiết này là lý do cách 1 cho đáp án **tối thiểu** (xem giải thích sau trace).

In [11]:
def candy_relaxation(ratings: List[int]) -> int:
    n = len(ratings)
    candies = [1] * n
    while True:
        changed = False
        for i in range(n):
            if i > 0 and ratings[i] > ratings[i - 1] and candies[i] <= candies[i - 1]:
                candies[i] = candies[i - 1] + 1
                changed = True
            if i < n - 1 and ratings[i] > ratings[i + 1] and candies[i] <= candies[i + 1]:
                candies[i] = candies[i + 1] + 1
                changed = True
        if not changed:
            return sum(candies)


print(candy_relaxation([1, 0, 2]))                  # 5
print(candy_relaxation([1, 2, 2]))                  # 4
print(candy_relaxation([1, 2, 87, 87, 87, 2, 1]))   # 13

5
4
13


### Trace cách 1 trên `[1, 6, 10, 8, 7, 3, 2]`

Hàm `trace_relaxation` in mảng `candies` **sau mỗi vòng** và những vị trí vừa được nâng. Hãy để ý: dốc xuống phía sau đỉnh `10` được "sửa" dần **từng bạn một mỗi vòng**, vì trong một vòng quét trái → phải, khi ta sửa bạn `i` thì bạn `i + 1` (bên phải) vẫn còn là giá trị cũ.

| vòng | candies sau vòng | vị trí được nâng |
|---:|---|---|
| 1 | `[1, 2, 3, 2, 2, 2, 1]` | 1, 2, 3, 4, 5 |
| 2 | `[1, 2, 3, 3, 3, 2, 1]` | 3, 4 |
| 3 | `[1, 2, 4, 4, 3, 2, 1]` | 2, 3 |
| 4 | `[1, 2, 5, 4, 3, 2, 1]` | 2 |
| 5 | `[1, 2, 5, 4, 3, 2, 1]` | (không ai) → dừng |

Đọc cột cuối từ trên xuống: chỗ "đang sửa" lùi dần về phía đỉnh `10` (index 2), mỗi vòng một bước. Đỉnh phải đợi tới vòng 4 mới biết mình cần 5 kẹo.

Tổng `1+2+5+4+3+2+1 = 18`.

In [12]:
def trace_relaxation(ratings):
    n = len(ratings)
    candies = [1] * n
    print("ratings        :", ratings)
    print("ban đầu        :", candies)
    rnd = 0
    while True:
        rnd += 1
        raised = []
        for i in range(n):
            if i > 0 and ratings[i] > ratings[i - 1] and candies[i] <= candies[i - 1]:
                candies[i] = candies[i - 1] + 1
                raised.append(i)
            if i < n - 1 and ratings[i] > ratings[i + 1] and candies[i] <= candies[i + 1]:
                candies[i] = candies[i + 1] + 1
                raised.append(i)
        print(f"sau vòng {rnd}     : {candies}  nâng: {sorted(set(raised)) or 'không ai -> dừng'}")
        if not raised:
            break
    print("tổng =", sum(candies), "| hợp lệ?", is_valid(ratings, candies))


trace_relaxation([1, 6, 10, 8, 7, 3, 2])

ratings        : [1, 6, 10, 8, 7, 3, 2]
ban đầu        : [1, 1, 1, 1, 1, 1, 1]
sau vòng 1     : [1, 2, 3, 2, 2, 2, 1]  nâng: [1, 2, 3, 4, 5]
sau vòng 2     : [1, 2, 3, 3, 3, 2, 1]  nâng: [3, 4]
sau vòng 3     : [1, 2, 4, 4, 3, 2, 1]  nâng: [2, 3]
sau vòng 4     : [1, 2, 5, 4, 3, 2, 1]  nâng: [2]
sau vòng 5     : [1, 2, 5, 4, 3, 2, 1]  nâng: không ai -> dừng
tổng = 18 | hợp lệ? True


### Vì sao cách 1 cho đúng **số kẹo tối thiểu**?

Gọi `best` là một cách chia hợp lệ **ít kẹo nhất** (ta chưa biết nó, chỉ biết nó tồn tại). Ta chứng minh **bất biến** (invariant — điều luôn đúng ở mọi thời điểm):

> Ở mọi thời điểm, `candies[i] <= best[i]` với mọi `i`.

- Ban đầu `candies[i] = 1 <= best[i]` vì ai cũng ít nhất 1 kẹo. ✔
- Mỗi lần nâng, ví dụ do luật trái: ta đặt `candies[i] = candies[i-1] + 1`. Vì `best` hợp lệ và `ratings[i] > ratings[i-1]`, ta có `best[i] >= best[i-1] + 1 >= candies[i-1] + 1`. Vậy sau khi nâng vẫn `candies[i] <= best[i]`. ✔ (Luật phải tương tự.)

Khi thuật toán dừng, `candies` **hợp lệ** (không còn vi phạm) và từng phần tử `<= best`. Mà `best` là ít kẹo nhất, nên tổng của `candies` không thể nhỏ hơn → **bằng đúng** tối thiểu.

Thuật toán chắc chắn dừng vì kẹo chỉ tăng, và không bao giờ vượt `best[i]`.

Ý tưởng "**chỉ nâng khi bị bắt buộc, nâng vừa đủ**" sẽ xuất hiện lại ở mọi cách sau.

### Vấn đề của cách 1: quá chậm

Trường hợp xấu nhất là một dốc xuống dài, ví dụ `[5, 4, 3, 2, 1]`: mỗi vòng chỉ "đẩy" được thông tin đi thêm một bạn sang trái, nên cần khoảng `n` vòng, mỗi vòng `O(n)` → **`O(n²)`**.

In [13]:
def count_relaxation_rounds(ratings):
    n = len(ratings)
    candies = [1] * n
    rounds = 0
    while True:
        rounds += 1
        changed = False
        for i in range(n):
            if i > 0 and ratings[i] > ratings[i - 1] and candies[i] <= candies[i - 1]:
                candies[i] = candies[i - 1] + 1; changed = True
            if i < n - 1 and ratings[i] > ratings[i + 1] and candies[i] <= candies[i + 1]:
                candies[i] = candies[i + 1] + 1; changed = True
        if not changed:
            return rounds


for n in [5, 100, 1000]:
    desc = list(range(n, 0, -1))          # dốc xuống dài n bạn
    r = count_relaxation_rounds(desc)
    print(f"n={n:>5}: số vòng = {r:>5}, số bước ~ {r} * {n} = {r * n}")

n=    5: số vòng =     5, số bước ~ 5 * 5 = 25
n=  100: số vòng =   100, số bước ~ 100 * 100 = 10000


n= 1000: số vòng =  1000, số bước ~ 1000 * 1000 = 1000000


Với `n = 1000` đã cần khoảng một triệu bước; với `n = 20 000` sẽ là khoảng **400 triệu** bước — quá chậm.

**Cách cũ vướng ở đâu?** Thông tin "phía sau còn một dốc xuống dài" chỉ được truyền sang trái mỗi vòng một bạn, vì ta luôn quét **trái → phải**. Nhưng luật phải là luật nhìn **sang phải**; nó nên được xử lý bằng một lượt quét **phải → trái**!

## 8. Bước đột phá: tách hai luật, mỗi luật một chiều quét

Ta tách bài toán thành hai bài dễ:

- **Bài A (chỉ có luật trái):** chia ít kẹo nhất sao cho ai cao điểm hơn bạn **bên trái** thì nhiều kẹo hơn. Quét trái → phải là xong, như hàm `candy_left_only` ở mục 6. Gọi kết quả là `left`.
- **Bài B (chỉ có luật phải):** tương tự nhưng với bạn **bên phải**, quét phải → trái. Gọi kết quả là `right`.

`left[i]` có ý nghĩa rất cụ thể: nó bằng **độ dài dốc lên kết thúc tại `i`** (đếm cả bạn `i`). Ví dụ `ratings = [1, 3, 4, 5, 2]`:

- tại `i = 3` (điểm 5): đi ngược về trái, `5 > 4 > 3 > 1` → dốc lên dài 4 bạn → `left[3] = 4`.
- tại `i = 4` (điểm 2): `2 < 5` nên dốc lên chỉ có mình nó → `left[4] = 1`.

Đây là một **cận dưới** (lower bound) bắt buộc: nếu có 4 bạn tăng điểm liên tiếp kết thúc tại `i`, thì kẹo của họ phải tăng nghiêm ngặt, ít nhất `1, 2, 3, 4` → bạn `i` cần **ít nhất** 4 kẹo, trong **mọi** cách chia hợp lệ. `right[i]` cũng là cận dưới như vậy theo chiều ngược lại.

Cell dưới tính `left` bằng định nghĩa "đếm dốc lên" (đi ngược về trái từng bạn) — chậm `O(n²)` nhưng dễ hiểu — để kiểm chứng ý nghĩa trên.

In [14]:
def increasing_run_ending_at(ratings, i):
    """Độ dài dốc lên (tăng nghiêm ngặt) kết thúc tại i, đếm cả i."""
    length = 1
    j = i
    while j > 0 and ratings[j] > ratings[j - 1]:
        length += 1
        j -= 1
    return length


r = [1, 3, 4, 5, 2]
print("ratings          :", r)
print("dốc lên kết tại i:", [increasing_run_ending_at(r, i) for i in range(len(r))])
print("candy_left_only  :", candy_left_only(r))   # trùng nhau!

ratings          : [1, 3, 4, 5, 2]
dốc lên kết tại i: [1, 2, 3, 4, 1]
candy_left_only  : [1, 2, 3, 4, 1]


## 9. Cách 2 — Hai mảng `left`/`right`, hai lượt quét, lấy `max` (cách chính)

Ta có hai cận dưới cho mỗi bạn: `left[i]` (do luật trái bắt buộc) và `right[i]` (do luật phải bắt buộc). Bạn `i` phải thỏa **cả hai**, nên cần **ít nhất** `max(left[i], right[i])` kẹo. Ý tưởng táo bạo: **cho đúng bằng chừng đó**.

```text
candies[i] = max(left[i], right[i])
```

### Bước 1: lượt trái → phải dựng `left`

```text
left[0] = 1
left[i] = left[i-1] + 1   nếu ratings[i] > ratings[i-1]
left[i] = 1               nếu không (bằng hoặc thấp hơn: dốc lên bị "cắt", đếm lại từ 1)
```

In [15]:
def build_left(ratings):
    n = len(ratings)
    left = [1] * n
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            left[i] = left[i - 1] + 1
    return left


print(build_left([1, 2, 87, 87, 87, 2, 1]))   # [1, 2, 3, 1, 1, 1, 1]

[1, 2, 3, 1, 1, 1, 1]


### Bước 2: lượt phải → trái dựng `right`

Đối xứng hoàn toàn: đọc bạn `i + 1` bên phải, nên duyệt `i` từ `n - 2` lùi về `0`.

```text
right[n-1] = 1
right[i] = right[i+1] + 1   nếu ratings[i] > ratings[i+1]
right[i] = 1                nếu không
```

In [16]:
def build_right(ratings):
    n = len(ratings)
    right = [1] * n
    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            right[i] = right[i + 1] + 1
    return right


print(build_right([1, 2, 87, 87, 87, 2, 1]))  # [1, 1, 1, 1, 3, 2, 1]

[1, 1, 1, 1, 3, 2, 1]


### Bước 3: ghép bằng `max` — vì sao vừa đúng luật vừa ít nhất?

**(a) Ít nhất.** Như mục 8: trong mọi cách chia hợp lệ, bạn `i` cần `>= left[i]` và `>= right[i]`, nên `>= max(left[i], right[i])`. Ta cho đúng bằng cận dưới này, không thể ít hơn được nữa.

**(b) Đúng luật trái.** Giả sử `ratings[i] > ratings[i-1]`. Cần chứng minh `max(left[i], right[i]) > max(left[i-1], right[i-1])`.

- `left[i] = left[i-1] + 1 > left[i-1]` (theo công thức dựng `left`).
- Vì `ratings[i-1] < ratings[i]`, bạn `i-1` **không** cao hơn bạn bên phải mình, nên `right[i-1] = 1`. Mà `left[i] >= 2 > 1 = right[i-1]`.
- Vậy `left[i]` lớn hơn **cả hai** số của bạn `i-1`, tức `max(...)` của bạn `i` > `max(...)` của bạn `i-1`. ✔

**(c) Đúng luật phải.** Đối xứng y hệt (đổi vai `left` ↔ `right`). ✔

Tóm lại: **mỗi luật chỉ làm tăng một trong hai mảng**, và khi một luật có hiệu lực thì mảng còn lại của hàng xóm chắc chắn bằng 1, nên lấy `max` không bao giờ phá luật nào.

Ví dụ số cụ thể với `[1, 0, 2]`: `left = [1, 1, 2]`, `right = [2, 1, 1]`, `max = [2, 1, 2]` → tổng 5. Đúng Example 1!

In [17]:
def candy_two_arrays(ratings: List[int]) -> int:
    n = len(ratings)
    left = [1] * n
    for i in range(1, n):                    # lượt 1: luật trái
        if ratings[i] > ratings[i - 1]:
            left[i] = left[i - 1] + 1
    right = [1] * n
    for i in range(n - 2, -1, -1):           # lượt 2: luật phải
        if ratings[i] > ratings[i + 1]:
            right[i] = right[i + 1] + 1
    return sum(max(a, b) for a, b in zip(left, right))


print(candy_two_arrays([1, 0, 2]))                  # 5
print(candy_two_arrays([1, 2, 2]))                  # 4
print(candy_two_arrays([1, 3, 4, 5, 2]))            # 11
print(candy_two_arrays([1, 2, 87, 87, 87, 2, 1]))   # 13

5
4
11
13


### Trace cách 2

**Ví dụ 1: `ratings = [1, 3, 4, 5, 2]`**

| i | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| ratings | 1 | 3 | 4 | 5 | 2 |
| left (dốc lên tới i) | 1 | 2 | 3 | 4 | 1 |
| right (dốc xuống từ i) | 1 | 1 | 1 | 2 | 1 |
| **max** | 1 | 2 | 3 | **4** | 1 |

Tổng `= 11`. Ở đỉnh `i = 3`: `left = 4`, `right = 2` → dốc lên dài hơn quyết định.

**Ví dụ 2: `ratings = [1, 2, 87, 87, 87, 2, 1]`**

| i | 0 | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|---|
| ratings | 1 | 2 | 87 | 87 | 87 | 2 | 1 |
| left | 1 | 2 | 3 | 1 | 1 | 1 | 1 |
| right | 1 | 1 | 1 | 1 | 3 | 2 | 1 |
| **max** | 1 | 2 | **3** | **1** | **3** | 2 | 1 |

Tổng `= 13`. Ba bạn điểm 87 được `3, 1, 3`: bạn đầu là đỉnh dốc lên, bạn cuối là đỉnh dốc xuống, bạn giữa không bị luật nào ép nên chỉ 1 kẹo (dù cùng điểm cao nhất!).

Hàm `trace_two_arrays` in các bảng trên từ code thật.

In [18]:
def trace_two_arrays(ratings):
    left, right = build_left(ratings), build_right(ratings)
    best = [max(a, b) for a, b in zip(left, right)]
    w = 4
    print("i       :" + "".join(f"{i:>{w}}" for i in range(len(ratings))))
    print("ratings :" + "".join(f"{x:>{w}}" for x in ratings))
    print("left    :" + "".join(f"{x:>{w}}" for x in left))
    print("right   :" + "".join(f"{x:>{w}}" for x in right))
    print("max     :" + "".join(f"{x:>{w}}" for x in best), " -> tổng =", sum(best),
          "| hợp lệ?", is_valid(ratings, best))
    return best


trace_two_arrays([1, 3, 4, 5, 2])
print()
best = trace_two_arrays([1, 2, 87, 87, 87, 2, 1])
print()
draw_candies(best)

i       :   0   1   2   3   4
ratings :   1   3   4   5   2
left    :   1   2   3   4   1
right   :   1   1   1   2   1
max     :   1   2   3   4   1  -> tổng = 11 | hợp lệ? True

i       :   0   1   2   3   4   5   6
ratings :   1   2  87  87  87   2   1
left    :   1   2   3   1   1   1   1
right   :   1   1   1   1   3   2   1
max     :   1   2   3   1   3   2   1  -> tổng = 13 | hợp lệ? True

           #       #
       #   #       #   #
   #   #   #   #   #   #   #
   1   2   3   1   3   2   1   <- số kẹo


**Độ phức tạp cách 2** (`n = len(ratings)`):

- Time `O(n)`: ba vòng lặp tách rời, mỗi vòng `n` bước.
- Space `O(n)`: hai mảng `left`, `right`.

Đây là cách **nên nhớ nhất**: dễ hiểu, dễ chứng minh, dễ viết lại khi đi phỏng vấn.

---

## 10. Cách 3 — Một mảng, hai lượt quét

**Cách 2 còn thừa ở đâu?** Ta giữ cả `left` lẫn `right` chỉ để cuối cùng lấy `max`. Có thể lấy `max` **ngay trong lượt quét thứ hai**:

1. Lượt 1 (trái → phải): `candies` = đúng mảng `left`.
2. Lượt 2 (phải → trái): nếu `ratings[i] > ratings[i+1]` thì

   ```text
   candies[i] = max(candies[i], candies[i+1] + 1)
   ```

   - `candies[i]` lúc này vẫn là `left[i]` (chưa bị lượt 2 chạm vào).
   - `candies[i+1]` thì **đã xong** lượt 2, tức bằng `max(left[i+1], right[i+1])`. Vì `ratings[i+1] < ratings[i]` nên `left[i+1] = 1`, do đó `candies[i+1] = right[i+1]`, và `candies[i+1] + 1` chính là `right[i]`.

   Vậy câu lệnh trên tính đúng `max(left[i], right[i])` mà không cần mảng `right`.

**Đừng quên `max`!** Nếu viết `candies[i] = candies[i+1] + 1` (ghi đè), ta có thể **hạ** kẹo của một đỉnh đã được lượt 1 nâng cao, phá luật trái. Ta sẽ xem ví dụ cụ thể ở mục lỗi thường gặp.

In [19]:
def candy_one_array(ratings: List[int]) -> int:
    n = len(ratings)
    candies = [1] * n
    for i in range(1, n):                     # lượt 1: luật trái
        if ratings[i] > ratings[i - 1]:
            candies[i] = candies[i - 1] + 1
    for i in range(n - 2, -1, -1):            # lượt 2: luật phải, giữ max
        if ratings[i] > ratings[i + 1]:
            candies[i] = max(candies[i], candies[i + 1] + 1)
    return sum(candies)


print(candy_one_array([1, 0, 2]))                   # 5
print(candy_one_array([1, 2, 2]))                   # 4
print(candy_one_array([1, 6, 10, 8, 7, 3, 2]))      # 18

5
4
18


### Trace cách 3 trên `[1, 6, 10, 8, 7, 3, 2]`

Sau lượt 1: `candies = [1, 2, 3, 1, 1, 1, 1]`. Lượt 2 đi từ `i = 5` về `0`:

| i | so sánh | candies[i] cũ | candies[i+1] + 1 | candies[i] mới |
|---:|---|---:|---:|---:|
| 5 | 3 > 2 | 1 | 2 | 2 |
| 4 | 7 > 3 | 1 | 3 | 3 |
| 3 | 8 > 7 | 1 | 4 | 4 |
| 2 | 10 > 8 | 3 | 5 | **5** (dốc xuống thắng) |
| 1 | 6 > 10? không | 2 | – | 2 |
| 0 | 1 > 6? không | 1 | – | 1 |

Kết quả `[1, 2, 5, 4, 3, 2, 1]`, tổng 18.

In [20]:
def trace_one_array(ratings):
    n = len(ratings)
    candies = [1] * n
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            candies[i] = candies[i - 1] + 1
    print("sau lượt 1:", candies)
    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            old, cand = candies[i], candies[i + 1] + 1
            candies[i] = max(old, cand)
            print(f"  i={i}: {ratings[i]:>2} > {ratings[i+1]:>2} -> max({old}, {cand}) = {candies[i]}")
        else:
            print(f"  i={i}: {ratings[i]:>2} > {ratings[i+1]:>2}? không -> giữ {candies[i]}")
    print("sau lượt 2:", candies, "tổng =", sum(candies))


trace_one_array([1, 6, 10, 8, 7, 3, 2])

sau lượt 1: [1, 2, 3, 1, 1, 1, 1]
  i=5:  3 >  2 -> max(1, 2) = 2
  i=4:  7 >  3 -> max(1, 3) = 3
  i=3:  8 >  7 -> max(1, 4) = 4
  i=2: 10 >  8 -> max(3, 5) = 5
  i=1:  6 > 10? không -> giữ 2
  i=0:  1 >  6? không -> giữ 1
sau lượt 2: [1, 2, 5, 4, 3, 2, 1] tổng = 18


**Độ phức tạp cách 3:** Time `O(n)`, Space `O(n)` (một mảng `candies` thay vì hai). Vẫn `O(n)` bộ nhớ, nhưng nhẹ bằng một nửa cách 2.

Câu hỏi tiếp theo rất tự nhiên: **có cần mảng nào không?** Đề chỉ hỏi **tổng** số kẹo, không hỏi từng bạn được bao nhiêu…

---

## 11. Cách 4 — Một lượt quét, bộ nhớ `O(1)`: đếm dốc lên / dốc xuống / đỉnh

Đây là cách **sáng tạo** nhất. Ta đi một lượt từ trái sang phải, cộng dồn `total`, và chỉ nhớ 3 con số:

- `up`: độ dài dốc lên **hiện tại** (số bước "lên" liên tiếp vừa đi qua),
- `down`: độ dài dốc xuống **hiện tại** (số bạn đã đi qua sau đỉnh, trên dốc xuống),
- `peak`: độ dài dốc lên đã dẫn tới **đỉnh gần nhất** (để biết đỉnh đang có bao nhiêu kẹo).

### Kiến thức nhỏ trước khi vào thuật toán: thêm một bạn vào cuối dốc xuống

Xét dốc xuống có `d` bạn đứng sau đỉnh. Cách chia ít nhất cho họ là `d, d-1, ..., 2, 1` (bạn cuối dốc 1 kẹo). Khi có **thêm một bạn** thấp hơn đứng vào cuối dốc (dốc dài `d + 1`):

- bạn mới nhận **1** kẹo,
- **mỗi** bạn cũ trên dốc phải được **nâng thêm 1** (vì bạn sau họ vừa "cao" thêm một bậc): thêm `d` kẹo.

Vậy chi phí thêm là `1 + d`, hay viết theo độ dài **mới** `down = d + 1` thì là **`down`**. Còn đỉnh có cần nâng không? Xem tiếp ở dưới. Cell dưới kiểm chứng chi phí này bằng số: tổng `1 + 2 + ... + k` tăng thêm đúng `k` khi `k` tăng 1.

In [21]:
prev = 0
for down in range(1, 6):
    slope = list(range(down, 0, -1))            # dốc xuống dài down: [down, ..., 2, 1]
    total = sum(slope)                          # = down * (down + 1) // 2
    print(f"down={down}: dốc {slope!s:<16} tổng={total:>2}  tăng thêm={total - prev}")
    prev = total

down=1: dốc [1]              tổng= 1  tăng thêm=1
down=2: dốc [2, 1]           tổng= 3  tăng thêm=2
down=3: dốc [3, 2, 1]        tổng= 6  tăng thêm=3
down=4: dốc [4, 3, 2, 1]     tổng=10  tăng thêm=4
down=5: dốc [5, 4, 3, 2, 1]  tổng=15  tăng thêm=5


### Ba trường hợp khi đi từ bạn `i-1` sang bạn `i`

Khởi tạo `total = 1` (bạn 0 được 1 kẹo), `up = down = peak = 0`.

**1. Lên dốc** (`ratings[i] > ratings[i-1]`):
- `up += 1`, `down = 0`, `peak = up`.
- Bạn `i` được `up + 1` kẹo (bạn đầu dốc có 1, mỗi bậc thêm 1) → `total += up + 1`.

**2. Đất bằng** (`ratings[i] == ratings[i-1]`):
- Không luật nào ràng buộc hai bạn này → bạn `i` bắt đầu lại từ 1 kẹo: `total += 1`.
- Reset `up = down = peak = 0`: bạn `i` giống như một "đỉnh" mới cao 0 bậc (đang có `peak + 1 = 1` kẹo).

**3. Xuống dốc** (`ratings[i] < ratings[i-1]`):
- `up = 0`, `down += 1`.
- Bạn mới nhận 1 và cả `down - 1` bạn trước đó trên dốc nâng thêm 1: thêm `down` kẹo.
- **Đỉnh** đang có `peak + 1` kẹo (nếu trước đó nó đã bị nâng vì dốc xuống thì đang có `down` kẹo). Bạn đứng ngay sau đỉnh giờ có `down` kẹo, nên đỉnh cần **> `down`**:
  - nếu `peak >= down` (tức `peak + 1 > down`): đỉnh vẫn đủ, **không** nâng → `total += down`,
  - nếu `peak < down`: dốc xuống đã dài bằng/hơn dốc lên, đỉnh phải **nâng thêm 1** → `total += down + 1`.

Gộp lại thành một dòng quen thuộc:

```text
total += 1 + down - (1 if peak >= down else 0)
```

Đây chính là "`max(left, right)` ở đỉnh" của cách 2, chỉ là được tính **dần dần**: đỉnh nhận `max(peak, down) + 1` kẹo.

In [22]:
def candy_one_pass(ratings: List[int]) -> int:
    n = len(ratings)
    if n <= 1:
        return n
    total = 1                     # bạn 0
    up = down = peak = 0
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:          # lên dốc
            up += 1
            down = 0
            peak = up
            total += up + 1
        elif ratings[i] == ratings[i - 1]:       # đất bằng
            up = down = peak = 0
            total += 1
        else:                                    # xuống dốc
            up = 0
            down += 1
            total += 1 + down - (1 if peak >= down else 0)
    return total


print(candy_one_pass([1, 0, 2]))                    # 5
print(candy_one_pass([1, 2, 2]))                    # 4
print(candy_one_pass([1, 3, 4, 5, 2]))              # 11
print(candy_one_pass([1, 2, 87, 87, 87, 2, 1]))     # 13
print(candy_one_pass([1, 6, 10, 8, 7, 3, 2]))       # 18

5
4
11
13
18


### Trace cách 4 — ví dụ 1: `[1, 3, 4, 5, 2]`

Cột "kẹo ảo" là mảng kẹo **tưởng tượng** ứng với `total` hiện tại (thuật toán thật **không** lưu mảng này; trace dựng lại để bạn nhìn thấy).

| i | bước | up | down | peak | cộng thêm | total | kẹo ảo |
|---:|---|---:|---:|---:|---|---:|---|
| 0 | (bắt đầu) | 0 | 0 | 0 | – | 1 | `[1]` |
| 1 | 1 → 3 lên | 1 | 0 | 1 | `up+1 = 2` | 3 | `[1, 2]` |
| 2 | 3 → 4 lên | 2 | 0 | 2 | `3` | 6 | `[1, 2, 3]` |
| 3 | 4 → 5 lên | 3 | 0 | 3 | `4` | 10 | `[1, 2, 3, 4]` |
| 4 | 5 → 2 xuống | 0 | 1 | 3 | `1 + 1 - 1 = 1` (peak 3 ≥ 1, đỉnh không nâng) | 11 | `[1, 2, 3, 4, 1]` |

Kết quả 11. Dốc lên dài (3) "che" hoàn toàn dốc xuống ngắn (1).

In [23]:
def trace_one_pass(ratings):
    n = len(ratings)
    total, up, down, peak = 1, 0, 0, 0
    virtual = [1]                                   # chỉ để minh họa
    print(f"{'i':>2} {'bước':<12}{'up':>3}{'down':>5}{'peak':>5}{'+':>4}{'total':>6}  kẹo ảo")
    print(f"{0:>2} {'bắt đầu':<12}{up:>3}{down:>5}{peak:>5}{'':>4}{total:>6}  {virtual}")
    for i in range(1, n):
        a, b = ratings[i - 1], ratings[i]
        if b > a:
            up += 1; down = 0; peak = up
            add = up + 1
            virtual.append(up + 1)
            kind = f"{a}->{b} lên"
        elif b == a:
            up = down = peak = 0
            add = 1
            virtual.append(1)
            kind = f"{a}->{b} bằng"
        else:
            up = 0; down += 1
            raise_peak = peak < down
            add = 1 + down - (0 if raise_peak else 1)
            m = len(virtual)
            for j in range(m - (down - 1), m):      # nâng các bạn cũ trên dốc
                virtual[j] += 1
            if raise_peak:
                virtual[m - down] += 1              # nâng đỉnh
            virtual.append(1)
            kind = f"{a}->{b} xuống" + ("*" if raise_peak else "")
        total += add
        assert total == sum(virtual)
        print(f"{i:>2} {kind:<12}{up:>3}{down:>5}{peak:>5}{add:>4}{total:>6}  {virtual}")
    print("(* = đỉnh bị nâng thêm 1)  hợp lệ?", is_valid(ratings, virtual))
    return total


trace_one_pass([1, 3, 4, 5, 2])

 i bước         up down peak   + total  kẹo ảo
 0 bắt đầu       0    0    0         1  [1]
 1 1->3 lên      1    0    1   2     3  [1, 2]
 2 3->4 lên      2    0    2   3     6  [1, 2, 3]
 3 4->5 lên      3    0    3   4    10  [1, 2, 3, 4]
 4 5->2 xuống    0    1    3   1    11  [1, 2, 3, 4, 1]
(* = đỉnh bị nâng thêm 1)  hợp lệ? True


11

### Trace cách 4 — ví dụ 2: `[1, 2, 87, 87, 87, 2, 1]`

| i | bước | up | down | peak | cộng thêm | total | kẹo ảo |
|---:|---|---:|---:|---:|---|---:|---|
| 0 | (bắt đầu) | 0 | 0 | 0 | – | 1 | `[1]` |
| 1 | 1 → 2 lên | 1 | 0 | 1 | 2 | 3 | `[1, 2]` |
| 2 | 2 → 87 lên | 2 | 0 | 2 | 3 | 6 | `[1, 2, 3]` |
| 3 | 87 → 87 bằng | 0 | 0 | 0 | 1 | 7 | `[1, 2, 3, 1]` |
| 4 | 87 → 87 bằng | 0 | 0 | 0 | 1 | 8 | `[1, 2, 3, 1, 1]` |
| 5 | 87 → 2 xuống | 0 | 1 | 0 | `1 + 1 - 0 = 2` (peak 0 < 1, nâng đỉnh) | 10 | `[1, 2, 3, 1, 2, 1]` |
| 6 | 2 → 1 xuống | 0 | 2 | 0 | `1 + 2 - 0 = 3` (peak 0 < 2, nâng đỉnh) | 13 | `[1, 2, 3, 1, 3, 2, 1]` |

Điểm đáng chú ý:

- Hai bước "bằng" reset mọi thứ về 0. Sau đó bạn 87 cuối cùng (index 4) trở thành "đỉnh" với `peak = 0` — đỉnh có 0 bậc dốc lên, đang giữ 1 kẹo.
- Vì `peak = 0` nhỏ hơn mọi `down >= 1`, mỗi bước xuống dốc đều phải nâng cả đỉnh: đỉnh đi `1 → 2 → 3`.
- Dốc lên `1, 2, 3` ở bên trái đất bằng **không bị ảnh hưởng** — đất bằng như bức tường ngăn hai bên.

In [24]:
trace_one_pass([1, 2, 87, 87, 87, 2, 1])

 i bước         up down peak   + total  kẹo ảo
 0 bắt đầu       0    0    0         1  [1]
 1 1->2 lên      1    0    1   2     3  [1, 2]
 2 2->87 lên     2    0    2   3     6  [1, 2, 3]
 3 87->87 bằng   0    0    0   1     7  [1, 2, 3, 1]
 4 87->87 bằng   0    0    0   1     8  [1, 2, 3, 1, 1]
 5 87->2 xuống*  0    1    0   2    10  [1, 2, 3, 1, 2, 1]
 6 2->1 xuống*   0    2    0   3    13  [1, 2, 3, 1, 3, 2, 1]
(* = đỉnh bị nâng thêm 1)  hợp lệ? True


13

### Trace cách 4 — ví dụ 3: dốc xuống dài hơn dốc lên `[1, 6, 10, 8, 7, 3, 2]`

Ví dụ này cho thấy chuyển biến giữa hai nhánh: khi `down` còn `<= peak = 2`, đỉnh không nâng; từ `down = 3` trở đi, đỉnh bị nâng mỗi bước.

| i | bước | down | peak | peak ≥ down? | cộng thêm | total |
|---:|---|---:|---:|---|---:|---:|
| 3 | 10 → 8 | 1 | 2 | có | 1 | 7 |
| 4 | 8 → 7 | 2 | 2 | có | 2 | 9 |
| 5 | 7 → 3 | 3 | 2 | **không** → nâng đỉnh | 4 | 13 |
| 6 | 3 → 2 | 4 | 2 | **không** → nâng đỉnh | 5 | 18 |

In [25]:
trace_one_pass([1, 6, 10, 8, 7, 3, 2])

 i bước         up down peak   + total  kẹo ảo
 0 bắt đầu       0    0    0         1  [1]
 1 1->6 lên      1    0    1   2     3  [1, 2]
 2 6->10 lên     2    0    2   3     6  [1, 2, 3]
 3 10->8 xuống   0    1    2   1     7  [1, 2, 3, 1]
 4 8->7 xuống    0    2    2   2     9  [1, 2, 3, 2, 1]
 5 7->3 xuống*   0    3    2   4    13  [1, 2, 4, 3, 2, 1]
 6 3->2 xuống*   0    4    2   5    18  [1, 2, 5, 4, 3, 2, 1]
(* = đỉnh bị nâng thêm 1)  hợp lệ? True


18

### Vì sao cách 4 đúng?

Chia hàng thành các "ngọn núi": mỗi ngọn gồm một dốc lên dài `u` bậc, đỉnh, rồi một dốc xuống dài `d` bậc (đất bằng tách các ngọn ra; đáy chung của hai ngọn liền nhau chỉ có 1 kẹo). Theo cách 2, trong một ngọn núi:

- các bạn trên dốc lên được `1, 2, ..., u`,
- các bạn trên dốc xuống được `d, ..., 2, 1`,
- đỉnh được `max(u, d) + 1`.

Cách 4 cộng **đúng các con số này**, chỉ là theo thứ tự thời gian: dốc lên cộng ngay `1..u` và đỉnh tạm có `u + 1`; mỗi bước xuống dốc cộng thêm `down` (kéo cả dốc lên 1), và chỉ khi `down > peak` thì cộng thêm 1 để đỉnh thành `down + 1`. Kết thúc, đỉnh có `max(peak, down) + 1`. Do đó `total` của cách 4 luôn bằng tổng của cách 2.

**Độ phức tạp:** Time `O(n)` (một lượt), Space **`O(1)`** (chỉ vài biến số nguyên).

## 12. Cách 5 (tùy chọn) — Sắp xếp theo điểm rồi phát kẹo

Một góc nhìn khác: **phát kẹo cho bạn điểm thấp trước**. Khi đến lượt bạn `i`, mọi hàng xóm có điểm **thấp hơn** bạn ấy đã được phát xong, nên ta biết chắc bạn `i` cần:

```text
candies[i] = 1 + max(kẹo của các hàng xóm có điểm thấp hơn)   (nếu không có ai thấp hơn thì 1)
```

### Kiến thức nền: `sorted` với `key` và `lambda`

- `sorted(range(n), key=lambda i: ratings[i])` trả về **danh sách index**, sắp theo điểm tăng dần.
- `lambda i: ratings[i]` là một **hàm vô danh** (anonymous function) viết gọn trên một dòng: nhận `i`, trả `ratings[i]`. `key=` bảo `sorted` so sánh theo giá trị đó thay vì so sánh chính `i`.

In [26]:
ratings = [1, 6, 10, 8, 7, 3, 2]
order = sorted(range(len(ratings)), key=lambda i: ratings[i])
print("index theo điểm tăng dần:", order)
print("điểm tương ứng          :", [ratings[i] for i in order])

index theo điểm tăng dần: [0, 6, 5, 1, 4, 3, 2]
điểm tương ứng          : [1, 2, 3, 6, 7, 8, 10]


Hai bạn **bằng điểm** đứng cạnh nhau thì không ai ràng buộc ai, nên thứ tự giữa họ không quan trọng. Còn hàng xóm **thấp hơn hẳn** luôn đứng **trước** trong `order`, nên đã có kẹo khi ta xét tới bạn `i`.

Đây là tư tưởng "**xử lý theo thứ tự phụ thuộc**" (gần với topological order — thứ tự tô-pô trong đồ thị): bạn `i` phụ thuộc vào hàng xóm thấp hơn, nên xử lý họ trước.

In [27]:
def candy_sorted(ratings: List[int]) -> int:
    n = len(ratings)
    candies = [1] * n
    for i in sorted(range(n), key=lambda k: ratings[k]):
        if i > 0 and ratings[i - 1] < ratings[i]:
            candies[i] = max(candies[i], candies[i - 1] + 1)
        if i < n - 1 and ratings[i + 1] < ratings[i]:
            candies[i] = max(candies[i], candies[i + 1] + 1)
    return sum(candies)


print(candy_sorted([1, 0, 2]))                  # 5
print(candy_sorted([1, 2, 2]))                  # 4
print(candy_sorted([1, 6, 10, 8, 7, 3, 2]))     # 18

5
4
18


**Độ phức tạp cách 5:** Time `O(n log n)` do sắp xếp, Space `O(n)`. Chậm hơn cách 2–4 một chút, nhưng ý tưởng này **tổng quát hơn**: nếu các bạn không đứng thành hàng mà đứng trên một **cây** hay một **đồ thị** bất kỳ (mỗi bạn có nhiều hàng xóm), hai lượt quét trái/phải không còn dùng được, nhưng "phát cho điểm thấp trước" vẫn chạy đúng.

---

## 13. Nộp LeetCode — `class Solution`

Bản chính dùng **cách 2** (hai mảng) vì dễ đọc và dễ chứng minh nhất. Bên dưới có thêm bản **cách 4** (một lượt, `O(1)` bộ nhớ) đặt tên `SolutionOnePass` để chạy trong notebook; khi nộp lên LeetCode, hãy **đổi tên lớp thành `Solution`**.

In [28]:
class Solution:
    def candy(self, ratings: List[int]) -> int:
        n = len(ratings)
        left = [1] * n
        for i in range(1, n):
            if ratings[i] > ratings[i - 1]:
                left[i] = left[i - 1] + 1
        right = [1] * n
        for i in range(n - 2, -1, -1):
            if ratings[i] > ratings[i + 1]:
                right[i] = right[i + 1] + 1
        return sum(max(a, b) for a, b in zip(left, right))


class SolutionOnePass:            # đổi tên thành Solution khi nộp
    def candy(self, ratings: List[int]) -> int:
        n = len(ratings)
        if n <= 1:
            return n
        total, up, down, peak = 1, 0, 0, 0
        for i in range(1, n):
            if ratings[i] > ratings[i - 1]:
                up += 1; down = 0; peak = up
                total += up + 1
            elif ratings[i] == ratings[i - 1]:
                up = down = peak = 0
                total += 1
            else:
                up = 0; down += 1
                total += 1 + down - (1 if peak >= down else 0)
        return total

In [29]:
tests = [
    ([1, 0, 2], 5),                     # Example 1
    ([1, 2, 2], 4),                     # Example 2
    ([7], 1),                           # n = 1
    ([2, 2, 2, 2], 4),                  # toàn bằng nhau
    ([1, 2, 3, 4, 5], 15),              # tăng dần
    ([5, 4, 3, 2, 1], 15),              # giảm dần
    ([2, 1], 3),                        # hai bạn, giảm
    ([1, 3, 4, 5, 2], 11),              # dốc lên dài hơn
    ([1, 6, 10, 8, 7, 3, 2], 18),       # dốc xuống dài hơn -> nâng đỉnh
    ([1, 2, 87, 87, 87, 2, 1], 13),     # có đất bằng ở đỉnh
    ([3, 2, 1, 2, 3], 11),              # thung lũng
    ([0, 20000], 3),                    # giá trị biên của ratings
]
for sol in (Solution(), SolutionOnePass()):
    for ratings, expected in tests:
        got = sol.candy(ratings)
        assert got == expected, (type(sol).__name__, ratings, got, expected)
    print(type(sol).__name__, "PASS", len(tests), "test")

Solution PASS 12 test
SolutionOnePass PASS 12 test


### Đối chiếu chéo tất cả các cách trên input ngẫu nhiên

Ta sinh nhiều mảng nhỏ ngẫu nhiên (cố định `random.seed` để chạy lại luôn ra cùng kết quả), dùng **vét cạn** làm trọng tài cho mảng rất nhỏ, và đối chiếu các cách nhanh với nhau trên mảng dài hơn. Điểm chỉ lấy trong `0..3` để có **nhiều trường hợp bằng nhau** — nơi dễ sai nhất.

In [30]:
import random

random.seed(135)
fast = [candy_relaxation, candy_two_arrays, candy_one_array, candy_one_pass, candy_sorted]

for _ in range(300):                                     # mảng rất nhỏ: so với vét cạn
    r = [random.randint(0, 3) for _ in range(random.randint(1, 6))]
    expected = candy_bruteforce(r)
    for f in fast:
        assert f(r) == expected, (f.__name__, r, f(r), expected)

for _ in range(2000):                                    # mảng dài hơn: so các cách với nhau
    r = [random.randint(0, 5) for _ in range(random.randint(1, 40))]
    expected = candy_two_arrays(r)
    for f in fast:
        assert f(r) == expected, (f.__name__, r)
    assert Solution().candy(r) == SolutionOnePass().candy(r) == expected

print("Tất cả", len(fast), "cách khớp vét cạn và khớp nhau trên 2300 input ngẫu nhiên!")

Tất cả 5 cách khớp vét cạn và khớp nhau trên 2300 input ngẫu nhiên!


## 14. Vì sao đúng — tóm tắt các bất biến

- **Cách 1 (relaxation):** luôn có `candies[i] <= best[i]` (chỉ nâng khi bị luật ép, nâng vừa đủ); dừng khi hợp lệ → bằng tối thiểu.
- **Cách 2 (hai mảng):** `left[i]` = độ dài dốc lên kết thúc tại `i`, `right[i]` = độ dài dốc xuống bắt đầu tại `i`. Cả hai là **cận dưới bắt buộc**. Lấy `max` đạt đúng cận dưới, và không phá luật vì khi luật trái có hiệu lực tại `i` thì `right[i-1] = 1` (và ngược lại).
- **Cách 3 (một mảng):** sau lượt 2 đã xử lý tới `i`, `candies[j] = max(left[j], right[j])` với mọi `j >= i`.
- **Cách 4 (một lượt):** sau khi xử lý tới `i`, `total` = tổng kẹo tối thiểu của đoạn `ratings[0..i]` (xét như một hàng riêng); đỉnh gần nhất đang giữ `max(peak, down) + 1`.
- **Cách 5 (sắp xếp):** khi xử lý bạn `i`, mọi hàng xóm thấp hơn đã có số kẹo cuối cùng, nên `1 + max(...)` là đúng cận dưới.

## 15. So sánh các cách

`n = len(ratings)`.

| Cách | Ý tưởng | Time | Space | Nhận xét |
|---|---|---:|---:|---|
| 0. Vét cạn | Thử mọi cách chia `1..n` kẹo | `O(n^n · n)` | `O(n)` | Chỉ làm trọng tài cho `n <= 6` |
| 1. Relaxation | Sửa vi phạm tới khi ổn định | `O(n²)` | `O(n)` | Cách người mới tự nghĩ, TLE với dốc dài |
| 2. Hai mảng | `max(left[i], right[i])` | `O(n)` | `O(n)` | **Nên nộp / nên nhớ**, dễ chứng minh |
| 3. Một mảng | Lượt 2 lấy `max` tại chỗ | `O(n)` | `O(n)` | Gọn hơn cách 2 |
| 4. Một lượt | Đếm `up` / `down` / `peak` | `O(n)` | `O(1)` | Tối ưu bộ nhớ, khó nghĩ ra hơn |
| 5. Sắp xếp | Phát cho điểm thấp trước | `O(n log n)` | `O(n)` | Tổng quát cho cây / đồ thị |

Lộ trình suy nghĩ: **sửa dần** (chậm vì thông tin lan từng bước) → nhận ra mỗi luật chỉ nhìn **một phía** → **mỗi phía một lượt quét** → ghép bằng `max` → bỏ mảng phụ → bỏ luôn cả mảng, chỉ nhớ **hình dạng con dốc**.

Cell dưới đo thời gian thật (con số tùy máy, chỉ cần nhìn tỉ lệ).

In [31]:
import time

def timed(f, arg):
    t = time.perf_counter()
    f(arg)
    return time.perf_counter() - t

random.seed(0)
big = [random.randint(0, 20000) for _ in range(20000)]
for f in (candy_two_arrays, candy_one_array, candy_one_pass, candy_sorted):
    print(f"{f.__name__:<18} n=20000 ngẫu nhiên : {timed(f, big) * 1000:7.2f} ms")

desc = list(range(2000, 0, -1))           # dốc xuống dài: ca xấu của relaxation
print(f"{'candy_relaxation':<18} n=2000  giảm dần  : {timed(candy_relaxation, desc) * 1000:7.2f} ms")
print(f"{'candy_two_arrays':<18} n=2000  giảm dần  : {timed(candy_two_arrays, desc) * 1000:7.2f} ms")

candy_two_arrays   n=20000 ngẫu nhiên :    4.46 ms
candy_one_array    n=20000 ngẫu nhiên :    2.64 ms
candy_one_pass     n=20000 ngẫu nhiên :    1.94 ms
candy_sorted       n=20000 ngẫu nhiên :   11.24 ms


candy_relaxation   n=2000  giảm dần  :  636.75 ms
candy_two_arrays   n=2000  giảm dần  :    0.30 ms


## 16. Lỗi người mới hay gặp

### Lỗi 1: Chỉ quét một chiều

Chỉ dùng lượt trái → phải (`candy_left_only`) → quên luật phải. `[1, 0, 2]` ra `[1, 1, 2]`, tổng 4, sai luật.

### Lỗi 2: Bằng điểm thì cho bằng kẹo

```python
elif ratings[i] == ratings[i - 1]:
    candies[i] = candies[i - 1]      # SAI: đề không yêu cầu bằng kẹo
```

Vẫn hợp lệ nhưng **không tối thiểu**: `[1, 2, 2]` ra 5 thay vì 4.

### Lỗi 3: Lượt 2 ghi đè thay vì lấy `max`

```python
candies[i] = candies[i + 1] + 1      # SAI: có thể hạ kẹo của đỉnh
```

Với `[1, 3, 4, 5, 2]`: sau lượt 1 `candies = [1, 2, 3, 4, 1]`; lượt 2 tại `i = 3` ghi đè `candies[3] = 2` → `[1, 2, 3, 2, 1]`, bạn điểm 5 lại ít kẹo hơn bạn điểm 4. Sửa: `candies[i] = max(candies[i], candies[i + 1] + 1)`.

### Lỗi 4: Vòng lặp ngược bỏ sót index 0

`for i in range(n - 2, 0, -1)` dừng ở `i = 1`. Với `[2, 1]`, bạn 0 không bao giờ được nâng → tổng 2 thay vì 3. Sửa: `range(n - 2, -1, -1)`.

### Lỗi 5: Cách 4 quên điều chỉnh đỉnh

- Luôn `total += 1 + down` (quên trừ 1 khi `peak >= down`) → đếm **thừa**: `[1, 3, 4, 5, 2]` ra 12 thay vì 11.
- Luôn `total += down` (không bao giờ nâng đỉnh) → đếm **thiếu**: `[1, 6, 10, 8, 7, 3, 2]` ra 16 thay vì 18.

### Lỗi 6: Dùng `ratings[i - 1]` khi `i = 0`

Python không báo lỗi với `ratings[-1]` mà so với bạn **cuối hàng** — như thể hàng là vòng tròn. Luôn cho vòng lặp bắt đầu từ `1` hoặc kèm `i > 0`.

Cell dưới chạy các bản sai để thấy tận mắt.

In [32]:
def wrong_equal_gets_equal(ratings):           # Lỗi 2
    n = len(ratings)
    c = [1] * n
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            c[i] = c[i - 1] + 1
        elif ratings[i] == ratings[i - 1]:
            c[i] = c[i - 1]
    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            c[i] = max(c[i], c[i + 1] + 1)
    return sum(c)

def wrong_overwrite(ratings):                  # Lỗi 3
    n = len(ratings)
    c = [1] * n
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            c[i] = c[i - 1] + 1
    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            c[i] = c[i + 1] + 1                # SAI: ghi đè
    return c

def wrong_skip_index0(ratings):                # Lỗi 4
    n = len(ratings)
    c = [1] * n
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            c[i] = c[i - 1] + 1
    for i in range(n - 2, 0, -1):              # SAI: thiếu i = 0
        if ratings[i] > ratings[i + 1]:
            c[i] = max(c[i], c[i + 1] + 1)
    return sum(c)

def wrong_one_pass(ratings, mode):             # Lỗi 5
    total, up, down, peak = 1, 0, 0, 0
    for i in range(1, len(ratings)):
        if ratings[i] > ratings[i - 1]:
            up += 1; down = 0; peak = up; total += up + 1
        elif ratings[i] == ratings[i - 1]:
            up = down = peak = 0; total += 1
        else:
            up = 0; down += 1
            total += (1 + down) if mode == "never_subtract" else down
    return total

c = candy_left_only([1, 0, 2])
print("Lỗi 1 [1,0,2]          :", c, "tổng", sum(c), "hợp lệ?", is_valid([1, 0, 2], c))
print("Lỗi 2 [1,2,2]          :", wrong_equal_gets_equal([1, 2, 2]), "(đúng: 4)")
c = wrong_overwrite([1, 3, 4, 5, 2])
print("Lỗi 3 [1,3,4,5,2]      :", c, "hợp lệ?", is_valid([1, 3, 4, 5, 2], c))
print("Lỗi 4 [2,1]            :", wrong_skip_index0([2, 1]), "(đúng: 3)")
print("Lỗi 5a [1,3,4,5,2]     :", wrong_one_pass([1, 3, 4, 5, 2], "never_subtract"), "(đúng: 11)")
print("Lỗi 5b [1,6,10,8,7,3,2]:", wrong_one_pass([1, 6, 10, 8, 7, 3, 2], "never_raise"), "(đúng: 18)")

Lỗi 1 [1,0,2]          : [1, 1, 2] tổng 4 hợp lệ? False
Lỗi 2 [1,2,2]          : 5 (đúng: 4)
Lỗi 3 [1,3,4,5,2]      : [1, 2, 3, 2, 1] hợp lệ? False
Lỗi 4 [2,1]            : 2 (đúng: 3)
Lỗi 5a [1,3,4,5,2]     : 12 (đúng: 11)
Lỗi 5b [1,6,10,8,7,3,2]: 16 (đúng: 18)


## 17. Mẹo nhận dạng pattern

- Mỗi phần tử bị ràng buộc với **hàng xóm hai bên**, và ràng buộc có **hướng** (lớn hơn / nhỏ hơn) → thử **tách thành hai ràng buộc một phía**: một lượt trái → phải, một lượt phải → trái, rồi **ghép** (bằng `max`, `min`, nhân, cộng… tùy bài).
- Đề hỏi "**tối thiểu**" và mỗi ràng buộc cho một **cận dưới** → thường đáp án là **max của các cận dưới**, nếu chứng minh được max đó vẫn hợp lệ.
- Thấy mình viết "**lặp tới khi không còn thay đổi**" (relaxation) mà mỗi vòng chỉ lan thông tin một bước → nghĩ xem có thể **chọn đúng thứ tự xử lý** (quét đúng chiều, hoặc sắp xếp) để mỗi phần tử chỉ cần xử lý **một lần** không.
- Chỉ quan trọng quan hệ `<`, `=`, `>` giữa hai phần tử liền kề, không quan trọng giá trị → vẽ thành **đồi dốc**, đếm **độ dài dốc**.
- Muốn giảm từ `O(n)` xuống `O(1)` bộ nhớ: tự hỏi "đề chỉ cần **tổng**, mình có thật sự cần nhớ từng phần tử không, hay chỉ cần vài con số tóm tắt trạng thái?".

## 18. Biến thể và bài liên quan

| Bài | Liên hệ |
|---|---|
| LeetCode 42 — Trapping Rain Water | Cũng dựng mảng "từ trái" và "từ phải" (max bên trái / bên phải) rồi ghép bằng `min` |
| LeetCode 238 — Product of Array Except Self | Prefix (từ trái) × suffix (từ phải) |
| LeetCode 845 — Longest Mountain in Array | Đếm độ dài dốc lên / dốc xuống, giống cách 4 |
| LeetCode 2100 — Find Good Days to Rob the Bank | Đếm độ dài dốc không tăng bên trái / không giảm bên phải bằng hai lượt quét |
| Biến thể tự chế: bằng điểm thì phải **bằng kẹo** | Thêm nhánh "copy" cho trường hợp bằng nhau ở cả hai lượt (Bài 5 bên dưới) |
| Biến thể tự chế: các bạn ngồi trên **cây** | Hai lượt quét không còn dùng được; cách 5 (phát cho điểm thấp trước) vẫn đúng |

## 19. Bài tập vận dụng

Làm theo thứ tự, từ dễ tới khó. Gợi ý ở mục 20, đáp án ở **cuối notebook**.

### Bài 1 — Tính tay (không chạy code)

Với mỗi mảng, lập bảng `left`, `right`, `max` rồi tính tổng kẹo:

- (a) `[1, 3, 2, 2, 1]`
- (b) `[4, 4, 1, 2]`
- (c) `[1, 2, 3, 3, 2, 1]`

Sau đó (tùy chọn) dùng `draw_hill` và `trace_two_arrays` để kiểm tra.

### Bài 2 — Đoán diễn biến của cách 4

Chạy tay `candy_one_pass([3, 2, 1, 2, 3])` (một **thung lũng**). Lập bảng `up`, `down`, `peak`, "cộng thêm", `total` cho từng `i`. Ở hai bước xuống dốc đầu tiên, đỉnh có bị nâng không? Vì sao? Kết quả cuối bằng bao nhiêu?

### Bài 3 — Điền TODO: cách 2 (hai mảng)

Hoàn thiện hàm theo 3 bước: dựng `left`, dựng `right`, cộng `max`.

In [33]:
def exercise_candy_two_arrays(ratings: List[int]) -> int:
    n = len(ratings)
    left = [1] * n
    right = [1] * n
    # TODO 1: vòng for i từ 1 tới n-1: nếu ratings[i] > ratings[i-1] thì left[i] = ...
    # TODO 2: vòng for i từ n-2 lùi về 0: nếu ratings[i] > ratings[i+1] thì right[i] = ...
    # TODO 3: trả về tổng max(left[i], right[i])
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_candy_two_arrays([1, 0, 2]) == 5
# assert exercise_candy_two_arrays([1, 2, 2]) == 4
# assert exercise_candy_two_arrays([1, 6, 10, 8, 7, 3, 2]) == 18

### Bài 4 — Tự viết lại: cách 3, trả về **danh sách kẹo**

Viết hàm trả về chính mảng `candies` (không phải tổng), chỉ dùng **một** mảng và **hai** lượt quét. Sau đó dùng `is_valid` để tự kiểm tra kết quả hợp lệ, và `sum(...)` phải bằng `candy_two_arrays`.

```python
exercise_candy_distribution([1, 2, 87, 87, 87, 2, 1]) -> [1, 2, 3, 1, 3, 2, 1]
```

In [34]:
def exercise_candy_distribution(ratings: List[int]) -> List[int]:
    # TODO: lượt 1 trái -> phải, lượt 2 phải -> trái (nhớ dùng max!)
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_candy_distribution([1, 2, 87, 87, 87, 2, 1]) == [1, 2, 3, 1, 3, 2, 1]
# assert exercise_candy_distribution([1, 0, 2]) == [2, 1, 2]
# assert is_valid([5, 3, 4, 1], exercise_candy_distribution([5, 3, 4, 1]))

### Bài 5 — Biến thể: bằng điểm thì phải **bằng kẹo**

Giữ hai luật cũ, thêm luật mới: nếu hai bạn **đứng cạnh nhau** có **cùng điểm** thì phải có **cùng số kẹo**. Tính số kẹo tối thiểu.

```python
candy_equal_same([1, 2, 2])    -> 5    # [1, 2, 2]
candy_equal_same([1, 2, 2, 1]) -> 6    # [1, 2, 2, 1]
candy_equal_same([3, 3, 1])    -> 5    # [2, 2, 1]
candy_equal_same([1, 0, 2])    -> 5    # không có cặp bằng nhau: như bài gốc
```

Câu hỏi tư duy: vì sao chỉ cần sửa **một dòng** trong mỗi lượt quét của cách 2 là đủ?

In [35]:
def exercise_candy_equal_same(ratings: List[int]) -> int:
    # TODO: giống cách 2, nhưng thêm nhánh cho trường hợp bằng điểm ở cả hai lượt
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_candy_equal_same([1, 2, 2]) == 5
# assert exercise_candy_equal_same([1, 2, 2, 1]) == 6
# assert exercise_candy_equal_same([3, 3, 1]) == 5
# assert exercise_candy_equal_same([1, 0, 2]) == 5

### Bài 6 — Mini challenge: một lượt bằng công thức tổng dốc

Một cách một lượt `O(1)` khác (cũng rất nổi tiếng): đi qua từng "ngọn núi", đếm `up` (số bước lên) và `down` (số bước xuống). Khi ngọn núi **kết thúc** (gặp đất bằng, hoặc đang xuống mà bắt đầu lên lại), cộng một lần:

```text
count(up) + count(down) + max(up, down)       với count(k) = 1 + 2 + ... + k = k(k+1)/2
```

rồi reset `up = down = 0`. Đất bằng cộng thêm 1. Cuối hàng cộng phần ngọn núi cuối cùng **và thêm 1**.

Yêu cầu: tự viết `exercise_candy_slope_formula`, giải thích vì sao `max(up, down)` xuất hiện, và vì sao đáy chung giữa hai ngọn núi không bị đếm hai lần.

In [36]:
def exercise_candy_slope_formula(ratings: List[int]) -> int:
    # TODO: count(k) = k*(k+1)//2; theo dõi old_slope / new_slope (+1, 0, -1)
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_candy_slope_formula([1, 0, 2]) == 5
# assert exercise_candy_slope_formula([1, 2, 2]) == 4
# assert exercise_candy_slope_formula([1, 2, 87, 87, 87, 2, 1]) == 13
# assert exercise_candy_slope_formula([1, 6, 10, 8, 7, 3, 2]) == 18

## 20. Gợi ý theo mức

**Bài 1**
- Mức 1: `left[i]` = độ dài dốc lên kết thúc tại `i`; `right[i]` = độ dài dốc xuống bắt đầu tại `i`.
- Mức 2: gặp "bằng" thì dốc bị cắt, đếm lại từ 1 — ở **cả hai** mảng.
- Mức 3: (a) `left = [1, 2, 1, 1, 1]` — tự làm `right` và `max`.

**Bài 2**
- Mức 1: bạn đầu tiên (điểm 3) là "đỉnh" với `peak = 0`.
- Mức 2: `peak = 0 < down` ở mọi bước xuống, nên đỉnh bị nâng mỗi bước.
- Mức 3: hai bước đầu cộng `2` rồi `3`; hai bước lên cộng `2` rồi `3`.

**Bài 3**
- Mức 1: chép lại cấu trúc `build_left` / `build_right`.
- Mức 2: vòng lùi là `for i in range(n - 2, -1, -1):`.
- Mức 3: `left[i] = left[i - 1] + 1`, `right[i] = right[i + 1] + 1`, `return sum(max(a, b) for a, b in zip(left, right))`.

**Bài 4**
- Mức 1: lượt 1 y hệt `left`.
- Mức 2: lượt 2 đọc `candies[i + 1]` (đã hoàn chỉnh) và so với `candies[i]` (mới chỉ là `left[i]`).
- Mức 3: `candies[i] = max(candies[i], candies[i + 1] + 1)`; cuối cùng `return candies`.

**Bài 5**
- Mức 1: hai bạn bằng điểm liền nhau giống như **dính** vào nhau thành một bạn.
- Mức 2: ở lượt trái: `elif ratings[i] == ratings[i - 1]: left[i] = left[i - 1]`; lượt phải đối xứng.
- Mức 3: vì cả `left` lẫn `right` đều "copy" trong một đoạn bằng nhau, nên `max` của chúng cũng bằng nhau trong đoạn đó → luật mới được giữ.

**Bài 6**
- Mức 1: một ngọn núi có dốc lên `up` bậc, dốc xuống `down` bậc: hai dốc được `1..up` và `1..down`, đỉnh được `max(up, down) + 1`.
- Mức 2: tổng thật của một ngọn núi là `count(up) + count(down) + max(up, down) + 1`. Công thức thiếu `1` — đó là kẹo của **chân trái** ngọn núi, vốn đã được đếm ở chỗ khác. Chỗ khác là đâu? Riêng bạn số 0 thì chưa ai đếm.
- Mức 3: ngọn núi kết thúc khi `(old_slope > 0 and new_slope == 0) or (old_slope < 0 and new_slope >= 0)`.

## 21. Đáp án tham khảo

Hãy tự làm trước rồi mới xem phần này.

**Bài 1**

| mảng | left | right | max | tổng |
|---|---|---|---|---:|
| (a) `[1,3,2,2,1]` | `[1,2,1,1,1]` | `[1,2,1,2,1]` | `[1,2,1,2,1]` | **7** |
| (b) `[4,4,1,2]` | `[1,1,1,2]` | `[1,2,1,1]` | `[1,2,1,2]` | **6** |
| (c) `[1,2,3,3,2,1]` | `[1,2,3,1,1,1]` | `[1,1,1,3,2,1]` | `[1,2,3,3,2,1]` | **12** |

Ở (c), hai bạn điểm 3 tình cờ cùng được 3 kẹo, nhưng vì hai lý do khác nhau: bạn trái do dốc lên, bạn phải do dốc xuống.

**Bài 2** — `[3, 2, 1, 2, 3]`

| i | bước | up | down | peak | cộng thêm | total |
|---:|---|---:|---:|---:|---|---:|
| 0 | bắt đầu | 0 | 0 | 0 | – | 1 |
| 1 | 3 → 2 xuống | 0 | 1 | 0 | `1 + 1 - 0 = 2` (nâng đỉnh) | 3 |
| 2 | 2 → 1 xuống | 0 | 2 | 0 | `1 + 2 - 0 = 3` (nâng đỉnh) | 6 |
| 3 | 1 → 2 lên | 1 | 0 | 1 | `2` | 8 |
| 4 | 2 → 3 lên | 2 | 0 | 2 | `3` | 11 |

Đỉnh (bạn 0) có `peak = 0` vì không có dốc lên nào dẫn tới nó, nên mỗi bước xuống dốc đều phải nâng đỉnh. Kẹo: `[3, 2, 1, 2, 3]`, tổng **11**.

**Bài 5 — vì sao một dòng là đủ:** một đoạn các bạn liền nhau cùng điểm phải cùng kẹo, nên có thể coi cả đoạn là **một bạn**. Ở lượt trái, "copy" `left[i] = left[i-1]` làm cả đoạn dùng chung cận dưới của bạn đầu đoạn (đúng vì cận dưới của đoạn là cận dưới của bạn đầu đoạn theo luật trái). Lượt phải đối xứng. `max` của hai giá trị chung cũng chung → luật bằng nhau được giữ, các luật cũ được giữ như chứng minh của cách 2.

**Bài 6 — vì sao `max(up, down)`:** đỉnh phải hơn cả bạn ngay trước nó trên dốc lên (`up` kẹo) lẫn bạn ngay sau nó trên dốc xuống (`down` kẹo), nên đỉnh cần `max(up, down) + 1`. Các bạn còn lại trên hai sườn được `1..up` và `1..down`. Vậy tổng thật của một ngọn núi là `count(up) + count(down) + max(up, down) + 1`.

**Vì sao công thức chỉ cộng `count(up) + count(down) + max(up, down)` (thiếu 1)?** Vì **chân trái** của mỗi ngọn núi (giữ 1 kẹo) đã được đếm ở chỗ khác: hoặc nó là đáy cuối của ngọn núi trước (nằm trong `count(down)` của ngọn trước), hoặc nó là bạn vừa được cộng `1` ở bước đất bằng. Nhờ bớt 1 cho mỗi ngọn, đáy chung không bị đếm hai lần. Chỉ riêng bạn số 0 là chân trái chưa ai đếm, nên cuối cùng cộng thêm `1`.

Ví dụ `[1, 6, 10, 8, 7, 3, 2]`: một ngọn núi duy nhất với `up = 2`, `down = 4` → `count(2) + count(4) + max(2, 4) + 1 = 3 + 10 + 4 + 1 = 18`.

Cell dưới kiểm chứng Bài 1, Bài 2 bằng máy, sau đó là đáp án code Bài 3–6.

In [37]:
# Kiểm chứng Bài 1 và Bài 2
assert trace_two_arrays([1, 3, 2, 2, 1]) == [1, 2, 1, 2, 1]
assert candy_two_arrays([4, 4, 1, 2]) == 6
assert candy_two_arrays([1, 2, 3, 3, 2, 1]) == 12
print()
assert trace_one_pass([3, 2, 1, 2, 3]) == 11

i       :   0   1   2   3   4
ratings :   1   3   2   2   1
left    :   1   2   1   1   1
right   :   1   2   1   2   1
max     :   1   2   1   2   1  -> tổng = 7 | hợp lệ? True

 i bước         up down peak   + total  kẹo ảo
 0 bắt đầu       0    0    0         1  [1]
 1 3->2 xuống*   0    1    0   2     3  [2, 1]
 2 2->1 xuống*   0    2    0   3     6  [3, 2, 1]
 3 1->2 lên      1    0    1   2     8  [3, 2, 1, 2]
 4 2->3 lên      2    0    2   3    11  [3, 2, 1, 2, 3]
(* = đỉnh bị nâng thêm 1)  hợp lệ? True


In [38]:
def answer_candy_two_arrays(ratings: List[int]) -> int:            # Bài 3
    n = len(ratings)
    left = [1] * n
    right = [1] * n
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            left[i] = left[i - 1] + 1
    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            right[i] = right[i + 1] + 1
    return sum(max(a, b) for a, b in zip(left, right))


def answer_candy_distribution(ratings: List[int]) -> List[int]:    # Bài 4
    n = len(ratings)
    candies = [1] * n
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            candies[i] = candies[i - 1] + 1
    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            candies[i] = max(candies[i], candies[i + 1] + 1)
    return candies


def answer_candy_equal_same(ratings: List[int]) -> int:            # Bài 5
    n = len(ratings)
    left = [1] * n
    right = [1] * n
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            left[i] = left[i - 1] + 1
        elif ratings[i] == ratings[i - 1]:
            left[i] = left[i - 1]                 # dòng mới
    for i in range(n - 2, -1, -1):
        if ratings[i] > ratings[i + 1]:
            right[i] = right[i + 1] + 1
        elif ratings[i] == ratings[i + 1]:
            right[i] = right[i + 1]               # dòng mới
    return sum(max(a, b) for a, b in zip(left, right))


def answer_candy_slope_formula(ratings: List[int]) -> int:         # Bài 6
    def count(k):
        return k * (k + 1) // 2

    n = len(ratings)
    if n <= 1:
        return n
    total = up = down = old_slope = 0
    for i in range(1, n):
        if ratings[i] > ratings[i - 1]:
            new_slope = 1
        elif ratings[i] < ratings[i - 1]:
            new_slope = -1
        else:
            new_slope = 0
        if (old_slope > 0 and new_slope == 0) or (old_slope < 0 and new_slope >= 0):
            total += count(up) + count(down) + max(up, down)       # chốt một ngọn núi
            up = down = 0
        if new_slope > 0:
            up += 1
        elif new_slope < 0:
            down += 1
        else:
            total += 1                                             # đất bằng
        old_slope = new_slope
    return total + count(up) + count(down) + max(up, down) + 1

In [39]:
# Test đáp án Bài 3–6
assert answer_candy_two_arrays([1, 0, 2]) == 5
assert answer_candy_two_arrays([1, 2, 2]) == 4
assert answer_candy_two_arrays([1, 6, 10, 8, 7, 3, 2]) == 18

assert answer_candy_distribution([1, 2, 87, 87, 87, 2, 1]) == [1, 2, 3, 1, 3, 2, 1]
assert answer_candy_distribution([1, 0, 2]) == [2, 1, 2]
assert is_valid([5, 3, 4, 1], answer_candy_distribution([5, 3, 4, 1]))

assert answer_candy_equal_same([1, 2, 2]) == 5
assert answer_candy_equal_same([1, 2, 2, 1]) == 6
assert answer_candy_equal_same([3, 3, 1]) == 5
assert answer_candy_equal_same([1, 0, 2]) == 5

assert answer_candy_slope_formula([1, 0, 2]) == 5
assert answer_candy_slope_formula([1, 2, 2]) == 4
assert answer_candy_slope_formula([1, 2, 87, 87, 87, 2, 1]) == 13
assert answer_candy_slope_formula([1, 6, 10, 8, 7, 3, 2]) == 18


def bruteforce_equal_same(ratings):
    """Trọng tài cho Bài 5: vét cạn với luật bằng điểm -> bằng kẹo."""
    n = len(ratings)
    best = None
    for c in itertools.product(range(1, n + 1), repeat=n):
        if not is_valid(ratings, c):
            continue
        if any(ratings[i] == ratings[i + 1] and c[i] != c[i + 1] for i in range(n - 1)):
            continue
        if best is None or sum(c) < best:
            best = sum(c)
    return best


random.seed(2024)
for _ in range(300):
    r = [random.randint(0, 3) for _ in range(random.randint(1, 6))]
    assert answer_candy_two_arrays(r) == candy_bruteforce(r)
    d = answer_candy_distribution(r)
    assert is_valid(r, d) and sum(d) == candy_bruteforce(r)
    assert answer_candy_equal_same(r) == bruteforce_equal_same(r), r
    assert answer_candy_slope_formula(r) == candy_bruteforce(r), r
for _ in range(2000):
    r = [random.randint(0, 5) for _ in range(random.randint(1, 40))]
    assert answer_candy_slope_formula(r) == candy_two_arrays(r), r
print("Đáp án tham khảo chạy PASS toàn bộ test!")

Đáp án tham khảo chạy PASS toàn bộ test!


## 22. Template ngắn nên nhớ

```python
class Solution:
    def candy(self, ratings: List[int]) -> int:
        n = len(ratings)
        left = [1] * n                              # luật trái: quét trái -> phải
        for i in range(1, n):
            if ratings[i] > ratings[i - 1]:
                left[i] = left[i - 1] + 1
        right = [1] * n                             # luật phải: quét phải -> trái
        for i in range(n - 2, -1, -1):
            if ratings[i] > ratings[i + 1]:
                right[i] = right[i + 1] + 1
        return sum(max(a, b) for a, b in zip(left, right))
```

Phiên bản `O(1)` bộ nhớ (cách 4), nhớ bằng câu "**lên thì cộng `up+1`, bằng thì cộng 1 và reset, xuống thì cộng `down` và nâng đỉnh nếu `peak < down`**":

```python
total, up, down, peak = 1, 0, 0, 0
for i in range(1, n):
    if ratings[i] > ratings[i-1]:   up += 1; down = 0; peak = up; total += up + 1
    elif ratings[i] == ratings[i-1]: up = down = peak = 0;        total += 1
    else:                            up = 0; down += 1;           total += down + (peak < down)
```

(`peak < down` là `True`/`False`; Python coi `True = 1`, `False = 0`, nên dòng cuối đúng bằng `1 + down - (1 if peak >= down else 0)`.)

Câu thần chú: **"Mỗi luật nhìn một phía → mỗi phía một lượt quét → lấy max của hai cận dưới."**

## 23. Checklist trước khi submit

- Mỗi bạn khởi tạo **1** kẹo (`[1] * n`)?
- Có xử lý **cả hai** luật (hai lượt quét, hoặc đếm cả dốc lên lẫn dốc xuống)?
- Bằng điểm thì **không** ép bằng kẹo, dốc bị cắt và đếm lại từ 1?
- Lượt 2 dùng **`max`**, không ghi đè?
- Vòng ngược là `range(n - 2, -1, -1)` (có index 0)?
- Không vô tình đọc `ratings[-1]` khi `i = 0`?
- Cách 4: có điều chỉnh đỉnh theo `peak >= down`, và xử lý `n = 1`?
- Đã thử: hai example của đề, `n = 1`, toàn bằng nhau, tăng dần, giảm dần, dốc xuống dài hơn dốc lên?